In [7]:
import yfinance as yf
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [8]:
tickers = ["CMG", "BB", "M", "QQQ"]
prices = {}

for ticker_symbol in tickers:
    ticker = yf.Ticker(ticker_symbol)
    prices[ticker_symbol] = ticker.history(start="2009-02-14", end="2020-06-11")
    print(f"\n{ticker_symbol}:")
    print(prices[ticker_symbol])


CMG:
                                Open       High        Low      Close  \
Date                                                                    
2009-02-17 00:00:00-05:00   1.097600   1.098800   1.057000   1.067200   
2009-02-18 00:00:00-05:00   1.069800   1.093000   1.048600   1.089000   
2009-02-19 00:00:00-05:00   1.106200   1.120000   1.068800   1.074800   
2009-02-20 00:00:00-05:00   1.054000   1.118000   1.048200   1.110000   
2009-02-23 00:00:00-05:00   1.121400   1.134200   1.086000   1.093400   
...                              ...        ...        ...        ...   
2020-06-04 00:00:00-04:00  20.886200  21.196199  20.700001  20.831200   
2020-06-05 00:00:00-04:00  21.000000  21.382000  20.901800  21.069201   
2020-06-08 00:00:00-04:00  21.056601  21.299999  20.664400  20.984600   
2020-06-09 00:00:00-04:00  20.738199  21.059799  20.700001  20.789400   
2020-06-10 00:00:00-04:00  20.852400  20.858801  20.469801  20.579201   

                             Volume  Dividen

In [9]:
#Collect Features, we want them a day behind, we give yesterdays stats to guess today's price, so score follows current day
for ticker_symbol in tickers:
    price = prices[ticker_symbol]

    price["Change"]      = price["Close"].shift(1) - price["Open"].shift(1)
    price["Score"]       = (price["Close"] > price["Open"]).astype(int)
    price["%Change"]     = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
    price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
    price["YestScore"]   = price["Score"].shift(1)
    price["5DayMean"]    = price["Close"].rolling(5).mean().shift(1)
    price["5DayVoli"]    = price["Close"].rolling(5).std().shift(1)
    price["5DayPerf"]    = price["Score"].rolling(5).sum().shift(1)

    price.dropna(inplace=True)
    prices[ticker_symbol] = price
    print(f"\n{ticker_symbol}:")
    print(price["5DayPerf"])


CMG:
Date
2009-02-24 00:00:00-05:00    2.0
2009-02-25 00:00:00-05:00    3.0
2009-02-26 00:00:00-05:00    2.0
2009-02-27 00:00:00-05:00    2.0
2009-03-02 00:00:00-05:00    2.0
                            ... 
2020-06-04 00:00:00-04:00    3.0
2020-06-05 00:00:00-04:00    3.0
2020-06-08 00:00:00-04:00    3.0
2020-06-09 00:00:00-04:00    2.0
2020-06-10 00:00:00-04:00    2.0
Name: 5DayPerf, Length: 2844, dtype: float64

BB:
Date
2009-02-24 00:00:00-05:00    0.0
2009-02-25 00:00:00-05:00    1.0
2009-02-26 00:00:00-05:00    2.0
2009-02-27 00:00:00-05:00    2.0
2009-03-02 00:00:00-05:00    3.0
                            ... 
2020-06-04 00:00:00-04:00    3.0
2020-06-05 00:00:00-04:00    4.0
2020-06-08 00:00:00-04:00    4.0
2020-06-09 00:00:00-04:00    4.0
2020-06-10 00:00:00-04:00    3.0
Name: 5DayPerf, Length: 2844, dtype: float64

M:
Date
2009-02-24 00:00:00-05:00    2.0
2009-02-25 00:00:00-05:00    2.0
2009-02-26 00:00:00-05:00    2.0
2009-02-27 00:00:00-05:00    2.0
2009-03-02 00:00:00-05

In [10]:

features = ["Change", "%Change", "CloseToOpen", "YestScore", "5DayMean", "5DayVoli", "5DayPerf"]
all_results = {}

for ticker_symbol in tickers:
    price = prices[ticker_symbol]
    priceTrain = price[price.index <= "2018-06-11"]
    priceTest = price[price.index > "2018-06-11"]

    results = []

    for i in range(len(priceTest)):
        xTrain = priceTrain[features]
        yTrain = priceTrain["Score"]

        scaler = StandardScaler()
        xTrain = scaler.fit_transform(xTrain)

        model = LogisticRegression()
        model.fit(xTrain, yTrain)

        # Grab yesterdays information to predict today's, data already transformed so i is yesterday
        yest = priceTest.iloc[[i]]
        yestScal = scaler.transform(yest[features])
        pred = model.predict(yestScal)[0]
        prob = model.predict_proba(yestScal)[0][1]
        print("Analyzing Day: ", priceTest.index[i])

        results.append({"Date": priceTest.index[i],
                        "Score": priceTest["Score"].iloc[i],
                        "Prediction": pred,
                        "Probability": prob,
                        "Open": priceTest["Open"].iloc[i],
                        "Close": priceTest["Close"].iloc[i]})

        priceTrain = pd.concat([priceTrain, yest])

    all_results[ticker_symbol] = pd.DataFrame(results)

Analyzing Day:  2018-06-12 00:00:00-04:00
Analyzing Day:  2018-06-13 00:00:00-04:00
Analyzing Day:  2018-06-14 00:00:00-04:00
Analyzing Day:  2018-06-15 00:00:00-04:00
Analyzing Day:  2018-06-18 00:00:00-04:00
Analyzing Day:  2018-06-19 00:00:00-04:00
Analyzing Day:  2018-06-20 00:00:00-04:00
Analyzing Day:  2018-06-21 00:00:00-04:00
Analyzing Day:  2018-06-22 00:00:00-04:00
Analyzing Day:  2018-06-25 00:00:00-04:00
Analyzing Day:  2018-06-26 00:00:00-04:00
Analyzing Day:  2018-06-27 00:00:00-04:00
Analyzing Day:  2018-06-28 00:00:00-04:00
Analyzing Day:  2018-06-29 00:00:00-04:00
Analyzing Day:  2018-07-02 00:00:00-04:00
Analyzing Day:  2018-07-03 00:00:00-04:00
Analyzing Day:  2018-07-05 00:00:00-04:00
Analyzing Day:  2018-07-06 00:00:00-04:00
Analyzing Day:  2018-07-09 00:00:00-04:00
Analyzing Day:  2018-07-10 00:00:00-04:00
Analyzing Day:  2018-07-11 00:00:00-04:00
Analyzing Day:  2018-07-12 00:00:00-04:00
Analyzing Day:  2018-07-13 00:00:00-04:00
Analyzing Day:  2018-07-16 00:00:0

In [11]:
for ticker_symbol, results in all_results.items():
    score = results["Score"]
    pred = results["Prediction"]
    op = results["Open"]
    close = results["Close"]

    accuracy  = accuracy_score(score, pred)
    precision = precision_score(score, pred, zero_division=0)
    recall    = recall_score(score, pred, zero_division=0)
    f1        = f1_score(score, pred, zero_division=0)

    print(f"\n{ticker_symbol}:")
    print(" Mean:",results["Score"].mean(), "Accuracy: ", accuracy, "\n  Precision: ", precision, "\n  Recall: ", recall, "\n  F1: ", f1)



CMG:
 Mean: 0.5009940357852882 Accuracy:  0.48111332007952284 
  Precision:  0.46511627906976744 
  Recall:  0.23809523809523808 
  F1:  0.31496062992125984

BB:
 Mean: 0.46322067594433397 Accuracy:  0.532803180914513 
  Precision:  0.375 
  Recall:  0.012875536480686695 
  F1:  0.024896265560165977

M:
 Mean: 0.46322067594433397 Accuracy:  0.5208747514910537 
  Precision:  0.48130841121495327 
  Recall:  0.44206008583690987 
  F1:  0.46085011185682323

QQQ:
 Mean: 0.5308151093439364 Accuracy:  0.5208747514910537 
  Precision:  0.5335051546391752 
  Recall:  0.7752808988764045 
  F1:  0.6320610687022901


# Comments
QQQ performs best with accuracy around 52% and F1 of 0.63, showing the model is picking up some signal but nothing strong. BB has extremely low recall (0.01) meaning it's almost never predicting up days. CMG and M are mediocre across the board. Generally the model isn't overfitting badly, but isn't extracting much signal either, suggesting the features alone aren't expressive enough. Sentiment data may improve this.

In [12]:
#Find out how much you would have made/lost with this method
for ticker_symbol, results in all_results.items():
    score = results["Score"]
    pred = results["Prediction"]
    op = results["Open"]
    close = results["Close"]

    totalIn = totalOut = totalInvested = totalStoodOut = alltradein = alltradeout = perftradein = perftradeout = 0

    for i, o, c, s in zip(pred.values, op.values, close.values, score.values):
        if i == 1:
            totalIn += 100
            totalInvested += 1
            temp = ((c - o)/o) * 100
            totalOut += 100 + temp
        else:
            totalStoodOut += 1
        alltradein += 100
        alltradeout += (((c-o)/o) * 100) + 100
        if s == 1:
            perftradein += 100
            perftradeout += (((c-o)/o) * 100) + 100

    print(f"\n{ticker_symbol}:")
    print(f"  Model:    Invested ${totalIn:.0f}, Returned ${totalOut:.2f}, Profit ${totalOut - totalIn:.2f} ({totalInvested} trades, {totalStoodOut} skipped)")
    print(f"  Buy&Hold: Invested ${alltradein:.0f}, Returned ${alltradeout:.2f}, Profit ${alltradeout - alltradein:.2f}")
    print(f"  Perfect:  Invested ${perftradein:.0f}, Returned ${perftradeout:.2f}, Profit ${perftradeout - perftradein:.2f}")


CMG:
  Model:    Invested $12900, Returned $12907.30, Profit $7.30 (129 trades, 374 skipped)
  Buy&Hold: Invested $50300, Returned $50346.79, Profit $46.79
  Perfect:  Invested $25200, Returned $25556.07, Profit $356.07

BB:
  Model:    Invested $800, Returned $795.70, Profit $-4.30 (8 trades, 495 skipped)
  Buy&Hold: Invested $50300, Returned $50223.65, Profit $-76.35
  Perfect:  Invested $23300, Returned $23692.35, Profit $392.35

M:
  Model:    Invested $21400, Returned $21371.31, Profit $-28.69 (214 trades, 289 skipped)
  Buy&Hold: Invested $50300, Returned $50171.24, Profit $-128.76
  Perfect:  Invested $23300, Returned $23774.62, Profit $474.62

QQQ:
  Model:    Invested $38800, Returned $38833.60, Profit $33.60 (388 trades, 115 skipped)
  Buy&Hold: Invested $50300, Returned $50325.43, Profit $25.43
  Perfect:  Invested $26700, Returned $26914.48, Profit $214.48


# Comments

Log reg beats or matches buy & hold on all 4 while making fewer trades. BB is rough but the model loses way less than just holding. Perfect ceiling is still pretty low at $300-600 on $50k, suggesting the data doesn't have much signal to extract regardless of model.